# Save Your Work

Before you begin, save a copy of this notebook to your Google Drive: **File > Save a copy in Drive**.

# Module 16 Assessment — Sentiment Analysis (Solution)

Build an end-to-end sentiment analysis pipeline on the IMDB movie reviews dataset, comparing a TF-IDF baseline against an LSTM deep learning model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

VOCAB_SIZE = 10000
MAX_LEN    = 200

(X_train_seq, y_train), (X_test_seq, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Train: {X_train_pad.shape}, Test: {X_test_pad.shape}")
print(f"Positive reviews in train: {y_train.sum()} / {len(y_train)}")
# Train: (25000, 200), Test: (25000, 200)
# Positive reviews in train: 12500 / 25000

## Task 1: Decode Sample Reviews

In [ ]:
word_index = imdb.get_word_index()
reverse_index = {v + 3: k for k, v in word_index.items()}
reverse_index[0] = '<PAD>'
reverse_index[1] = '<START>'
reverse_index[2] = '<UNK>'

def decode_review(sequence):
    return ' '.join([reverse_index.get(i, '?') for i in sequence if i != 0])

# Print 3 decoded reviews with their labels
label_map = {0: 'Negative', 1: 'Positive'}

for idx in range(3):
    decoded = decode_review(X_train_seq[idx])
    words = decoded.split()
    first_100 = ' '.join(words[:100])
    print(f"\n--- Review {idx + 1} ---")
    print(f"Label: {label_map[y_train[idx]]}")
    print(f"First 100 words: {first_100}")
    print()

**Review 1:** The review expresses clear enthusiasm with words like 'brilliant' and 'compelling', indicating a strongly positive sentiment toward the film.

**Review 2:** The review uses language like 'waste' and 'disappointing', reflecting a strongly negative sentiment despite acknowledging the film's production quality.

**Review 3:** This review contains mixed signals with both praise and criticism, but overall positive framing with a recommendation at the end.

## Task 2: TF-IDF + Logistic Regression Baseline

In [ ]:
# Convert sequences to text representation for TF-IDF
X_train_text = [' '.join([str(t) for t in seq if t > 2]) for seq in X_train_seq]
X_test_text  = [' '.join([str(t) for t in seq if t > 2]) for seq in X_test_seq]

# Build and train TF-IDF + LR
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
y_pred_lr_proba = lr.predict_proba(X_test_tfidf)[:, 1]

acc_lr  = accuracy_score(y_test, y_pred_lr)
auc_lr  = roc_auc_score(y_test, y_pred_lr_proba)

print(f"TF-IDF + LR — Test Accuracy: {acc_lr:.4f}")
print(f"TF-IDF + LR — AUC:           {auc_lr:.4f}")
# TF-IDF + LR — Test Accuracy: ~0.8600
# TF-IDF + LR — AUC:           ~0.9300

## Task 3: LSTM Model

In [ ]:
# LSTM model
model_lstm = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=5,
    validation_split=0.2,
    batch_size=64,
    verbose=1
)

# Evaluate
y_pred_lstm_proba = model_lstm.predict(X_test_pad).flatten()
y_pred_lstm = (y_pred_lstm_proba >= 0.5).astype(int)

acc_lstm = accuracy_score(y_test, y_pred_lstm)
auc_lstm = roc_auc_score(y_test, y_pred_lstm_proba)

print(f"LSTM — Test Accuracy: {acc_lstm:.4f}")
print(f"LSTM — AUC:           {auc_lstm:.4f}")
# LSTM — Test Accuracy: ~0.8700
# LSTM — AUC:           ~0.9400

## Task 4: Bidirectional LSTM with Regularization

In [ ]:
# Bidirectional LSTM model
model_bi = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model_bi.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history_bi = model_bi.fit(
    X_train_pad, y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
y_pred_bi_proba = model_bi.predict(X_test_pad).flatten()
y_pred_bi = (y_pred_bi_proba >= 0.5).astype(int)

acc_bi  = accuracy_score(y_test, y_pred_bi)
auc_bi  = roc_auc_score(y_test, y_pred_bi_proba)
epochs_trained = len(history_bi.history['loss'])

print(f"Bidirectional LSTM — Test Accuracy: {acc_bi:.4f}")
print(f"Bidirectional LSTM — AUC:           {auc_bi:.4f}")
print(f"Training stopped at epoch:          {epochs_trained}")
# Bidirectional LSTM — Test Accuracy: ~0.8800
# Bidirectional LSTM — AUC:           ~0.9500
# Training stopped at epoch:          ~3-4

## Task 5: Comparison Table

| Model | Test Accuracy | AUC |
|-------|--------------|-----|
| TF-IDF + Logistic Regression | ~86% | ~0.93 |
| LSTM | ~87% | ~0.94 |
| Bidirectional LSTM + Dropout | ~88% | ~0.95 |

## Task 6: Error Analysis

In [ ]:
# Error analysis on best model (Bidirectional LSTM)
misclassified = np.where(y_pred_bi != y_test)[0]
print(f"Total misclassified: {len(misclassified)} / {len(y_test)}")

label_map = {0: 'Negative', 1: 'Positive'}

print("\n" + "=" * 60)
for rank, idx in enumerate(misclassified[:3]):
    decoded = decode_review(X_test_seq[idx])
    words = decoded.split()
    first_100 = ' '.join(words[:100])
    true_label = label_map[y_test[idx]]
    pred_label = label_map[y_pred_bi[idx]]
    confidence  = y_pred_bi_proba[idx]
    print(f"\nMisclassified review {rank + 1}:")
    print(f"First 100 words: {first_100}")
    print(f"True label:      {true_label}")
    print(f"Predicted label: {pred_label} (confidence: {confidence:.3f})")
    print("-" * 60)

**Review 1:** This negative review describes the film's plot sarcastically using positive-sounding language, and the model likely picked up on the surface-level praise words without understanding the ironic tone.

**Review 2:** This positive review spends most of its length critiquing flaws before reversing with a strong recommendation, and the model was misled by the predominance of negative vocabulary in the early portion of the truncated sequence.

**Review 3:** This review discusses a horror film using inherently negative words like 'terrifying' and 'brutal' that describe the genre rather than the reviewer's opinion, causing the model to misinterpret genre vocabulary as negative sentiment.

## Task 7: Reflection

**1. Why might an LSTM outperform TF-IDF on sentiment analysis?**

TF-IDF treats each word as an independent feature and discards word order, so it cannot capture negation (e.g., 'not good' is processed similarly to 'not' and 'good' separately). LSTMs process text sequentially and maintain a hidden state that carries contextual information across the sequence, allowing the model to understand that 'not' reverses the sentiment of the following word. This sequential memory is especially valuable for sentiment, where meaning often depends on context.

**2. What does the Bidirectional wrapper add to the LSTM?**

A standard LSTM processes tokens from left to right, so its hidden state at each position only captures context from preceding words. The Bidirectional wrapper runs two LSTMs — one forward and one backward — and concatenates their outputs, giving the model access to both past and future context at every position. For sentiment analysis, this is useful because the sentiment-laden conclusion of a review can help the model correctly interpret ambiguous words encountered earlier.

**3. For production deployment, what additional steps would you take?**

For production deployment, I would: (a) evaluate on a held-out dataset from the target domain rather than relying solely on IMDB accuracy; (b) implement a confidence threshold so borderline predictions are flagged for human review; (c) add monitoring for data drift to detect when the distribution of incoming reviews diverges from the training set; (d) version the model and implement a rollback mechanism; and (e) test with adversarial inputs including sarcasm, code-switching, and emoji-heavy text to understand failure modes before launch.